# Running Qiskit Circuits with LabOne Q

Many quantum programs start out as [Qiskit](https://www.ibm.com/quantum/qiskit) circuits. LabOne Q runs them through [OpenQASM 3](https://openqasm.com/): Qiskit exports a circuit as an OpenQASM program, and the `OpenQASMTranspiler` of LabOne Q turns the program into an `Experiment` for your QPU.

This tutorial takes a Bell-state circuit from Qiskit to measurement results, and then shows parameterized circuits and circuits that address physical qubits.

To run this tutorial, install Qiskit:

```bash
pip install qiskit
```

## Setup

We generate a device setup with two qubits and connect to an emulated session. A qubit is a quantum element with the signals that the quantum operations below play on, and a calibration that sets the local oscillators of the instruments. Set `do_emulation=False` to run on your instruments.

In [ ]:
import attrs

from laboneq.contrib.example_helpers.generate_device_setup import generate_device_setup
from laboneq.simple import *


@attrs.define()
class MiniTransmon(QuantumElement):
    REQUIRED_SIGNALS = ("acquire", "drive", "flux", "measure")

    def calibration(self):
        drive_lo = Oscillator(frequency=6.0e9)
        readout_lo = Oscillator(frequency=7.0e9)
        return Calibration(
            {
                self.signals["drive"]: SignalCalibration(local_oscillator=drive_lo),
                self.signals["measure"]: SignalCalibration(local_oscillator=readout_lo),
                self.signals["acquire"]: SignalCalibration(local_oscillator=readout_lo),
            }
        )


setup = generate_device_setup(
    number_qubits=2,
    pqsc=[{"serial": "DEV10001"}],
    hdawg=[{"serial": "DEV8001", "number_of_channels": 8, "options": None}],
    shfqc=[
        {
            "serial": "DEV12001",
            "number_of_channels": 6,
            "readout_multiplex": 2,
            "options": None,
        }
    ],
    include_flux_lines=True,
    server_host="localhost",
    setup_name="qiskit_tutorial_setup",
)
qubits = MiniTransmon.from_device_setup(setup)
q0, q1 = qubits

session = Session(setup)
session.connect(do_emulation=True)

## Quantum operations

For every gate of a program, the transpiler calls the quantum operation of the same name, passing the qubits of the gate first and then its parameters.

Qiskit can transpile any circuit into the gates `rz`, `sx`, `x` and `cz`, so we implement these, together with `measure` and `reset`. The pulses below are placeholders: replace them with the calibrated gates of your QPU. `rz` is a virtual gate, which shifts the phase of the drive pulses that follow it.

In [ ]:
from laboneq.dsl.quantum import QPU


class QasmOperations(dsl.QuantumOperations):
    QUBIT_TYPES = MiniTransmon

    @dsl.quantum_operation
    def sx(self, q):
        pulse = dsl.pulse_library.drag(length=50e-9, amplitude=0.25)
        dsl.play(q.signals["drive"], pulse)

    @dsl.quantum_operation
    def x(self, q):
        pulse = dsl.pulse_library.drag(length=50e-9, amplitude=0.5)
        dsl.play(q.signals["drive"], pulse)

    @dsl.quantum_operation
    def rz(self, q, angle):
        dsl.play(q.signals["drive"], pulse=None, increment_oscillator_phase=angle)

    @dsl.quantum_operation
    def cz(self, control, target):
        pulse = dsl.pulse_library.gaussian_square(length=100e-9, amplitude=0.3)
        dsl.play(control.signals["flux"], pulse)

    @dsl.quantum_operation
    def measure(self, q, handle):
        pulse = dsl.pulse_library.const(length=1e-6, amplitude=0.5)
        dsl.play(q.signals["measure"], pulse)
        dsl.acquire(q.signals["acquire"], handle=handle, length=1e-6)

    @dsl.quantum_operation
    def reset(self, q):
        dsl.delay(q.signals["drive"], time=10e-6)


qpu = QPU(qubits, quantum_operations=QasmOperations())

Create the transpiler for the QPU:

In [ ]:
from laboneq import openqasm3

transpiler = openqasm3.OpenQASMTranspiler(qpu)

## From a Qiskit circuit to results

Build a Bell-state circuit in Qiskit. `measure_all` adds a barrier before the measurements, so that both qubits are measured together, as qubits on a multiplexed readout line must be.

In [ ]:
from qiskit import QuantumCircuit, qasm3, transpile

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()
bell.draw()

Transpile the circuit into the gates of our quantum operations and export it as an OpenQASM 3 program:

In [ ]:
native_gates = ["rz", "sx", "x", "cz"]

program = qasm3.dumps(transpile(bell, basis_gates=native_gates))
print(program)

Map the qubit register `q` of the program to the qubits of the QPU and turn the program into an experiment. Then compile and run it like any other LabOne Q experiment:

In [ ]:
experiment = transpiler.experiment(
    program=program,
    qubit_map={"q": [q0, q1]},
    options=openqasm3.SingleProgramOptions(count=1024),
)
compiled_experiment = session.compile(experiment)
results = session.run(compiled_experiment)

Each measurement stores its result under the classical bit it assigns, here `meas[0]` and `meas[1]`:

In [ ]:
for handle in ["meas[0]", "meas[1]"]:
    print(handle, results.acquired_results[handle].data)

## Parameterized circuits

Qiskit exports the parameters of a circuit as inputs of the program. Pass their values with `inputs`:

In [ ]:
import numpy as np
from qiskit.circuit import Parameter

theta = Parameter("theta")
rotation = QuantumCircuit(1)
rotation.ry(theta, 0)
rotation.measure_all()

program = qasm3.dumps(transpile(rotation, basis_gates=native_gates))
print(program)

In [ ]:
for value in np.linspace(0, np.pi, 3):
    experiment = transpiler.experiment(
        program=program, qubit_map={"q": [q0]}, inputs={"theta": value}
    )
    results = session.run(session.compile(experiment))
    print(f"theta = {value:.2f}:", results.acquired_results["meas[0]"].data)

## Physical qubits

When Qiskit transpiles a circuit for a coupling map, the program addresses physical qubits, such as `$0` and `$1`. Map them to the qubits of your QPU in the same way:

In [ ]:
program = qasm3.dumps(
    transpile(
        bell,
        basis_gates=native_gates,
        coupling_map=[[0, 1], [1, 0]],
        initial_layout=[1, 0],
    )
)
print(program)

experiment = transpiler.experiment(program=program, qubit_map={"$0": q0, "$1": q1})
results = session.run(session.compile(experiment))

## Gates outside the standard library

The transpiler does not support gate definitions, and Qiskit writes one for every gate outside the OpenQASM standard library, `stdgates.inc`, such as `iswap`. Transpile such circuits into the gates of your quantum operations as above, or name the gate as a basis gate when you export the circuit, so that Qiskit does not define it, and implement a quantum operation of that name:

```python
program = qasm3.dumps(circuit, basis_gates=["U", "iswap"])
```

## Next steps

* [OpenQASM with LabOne Q](00_program_to_experiment.ipynb) covers the transpiler in detail, including qubit registers, gate broadcasting, inputs, delays and barriers.
* The how-to guides of the Applications Library run a [variational quantum eigensolver](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/04_qasm/01_VQE_Qiskit.html) and [randomized benchmarking](https://docs.zhinst.com/labone_q_user_manual/applications_library/how-to-guides/sources/04_qasm/02_RandomizedBenchmarking_from_Qiskit.html) from Qiskit on tunable transmons.